# 29 — B1 feature aggregate + downstream-ready delta flat bundle

Notebook này là gate một lần, chạy **CPU** sau khi notebook 25–28 của cả Quang
và Trọng đều `COMPLETE`. Nó không chạy lại model và không đọc held-out.

Nó xác minh fail-closed 8 runner manifest, mọi marker và hash artifact; lọc đúng
5.301 frame B1; sau đó tạo một delta flat bundle gồm SigLIP, EasyOCR,
PaddleOCR, VietOCR, Qwen caption/OCR và BGE-M3. Bundle này là đầu vào bất biến
cho notebook 30 chạy B0–B1 DEV A/B.

Đây **không** phải production merge và không sửa volume `keyframes-btc`. Vì
baseline BTC không nằm trong notebook này, SQLite snapshot và index B0+B1 chỉ
được dựng tạm ở notebook 30 sau khi attach baseline.

Thành công khi dòng cuối có `B1_FEATURE_AGGREGATE: COMPLETE`,
`feature_frames: 5301`, `siglip_rows: 5301`, `bge_rows: 5301`.


## 0 — Khóa lượt aggregate


In [ ]:
from pathlib import Path

RUN_SCOPE = "AGGREGATE"
CONFIRM_AGGREGATE = "B1"
DOWNLOAD_WORKERS = 16

EXPECTED_B1_COUNT = 5301
EXPECTED_PARITY_FREEZE_SHA256 = "3b9993755bb19000e80a9c7bd9ccdde1e9a4648dd1950167d0759f9a14227180"
EXPECTED_NB24_COMPLETE_SHA256 = "39950f697a1bfa9d6e6c9b78e220b8b62da48a99d8465b6f9b46a9c6b16a5c10"
SIGLIP_DIM = 1152
BGE_DIM = 1024
NORM_ATOL = 2e-3
RUNNERS = ("quang", "trong")

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/b1-feature-aggregate")
INPUT = WORK / "input"
OUTPUT = WORK / "output"
FLAT = OUTPUT / "delta-flat" / "data" / "vectors"

if RUN_SCOPE != "AGGREGATE" or CONFIRM_AGGREGATE != "B1":
    raise RuntimeError("Aggregate lock is closed; use RUN_SCOPE='AGGREGATE' and CONFIRM_AGGREGATE='B1'")
for path in (WORK, INPUT, OUTPUT, FLAT):
    path.mkdir(parents=True, exist_ok=True)


## 1 — Contract helpers


In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from typing import Any

import numpy as np


TEXT_SPECS = {
    "easyocr": {
        "root": "gate1-b1-easyocr-v1",
        "stages": ("n5_ocr_easyocr",),
        "artifacts": ("ocr_text",),
    },
    "qwen25": {
        "root": "gate1-b1-qwen25-v1",
        "stages": ("n6_caption_qwen", "n6_ocr_qwen25"),
        "artifacts": ("qwen_captions", "qwen_ocr"),
    },
    "paddle_vietocr": {
        "root": "gate1-b1-paddle-vietocr-v1",
        "stages": ("n5_ocr_paddle", "n5_ocr_vietocr"),
        "artifacts": ("paddle_ocr", "viet_ocr"),
    },
}


def signed_payload(payload: dict[str, Any], field: str, hash_json) -> dict[str, Any]:
    unsigned = dict(payload)
    claimed = unsigned.pop(field, None)
    if not isinstance(claimed, str) or claimed != hash_json(unsigned):
        raise RuntimeError(f"Invalid signed payload field: {field}")
    return unsigned


def split_b1_rows(rows: list[dict[str, Any]]) -> tuple[dict[str, dict[str, list[dict[str, Any]]]], dict[str, int]]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        grouped[str(row["video_id"])].append(row)
    loads = {runner: 0 for runner in RUNNERS}
    owner: dict[str, str] = {}
    for video_id, video_rows in sorted(grouped.items(), key=lambda item: (-len(item[1]), item[0])):
        runner = min(RUNNERS, key=lambda value: (loads[value], value))
        owner[video_id] = runner
        loads[runner] += len(video_rows)
    partitions = {runner: {} for runner in RUNNERS}
    for video_id, video_rows in grouped.items():
        partitions[owner[video_id]][video_id] = sorted(video_rows, key=lambda row: int(row["frame_idx"]))
    return partitions, loads


def validate_runner_manifest(
    manifest: dict[str, Any], *, pipeline: str, runner: str,
    expected_videos: int, expected_frames: int, hash_json,
) -> None:
    signed_payload(manifest, "manifest_payload_hash", hash_json)
    checks = {
        "status": manifest.get("status") == "COMPLETE",
        "pipeline": manifest.get("pipeline") == pipeline,
        "runner": manifest.get("runner") == runner,
        "assigned_videos": manifest.get("assigned_videos") == expected_videos,
        "verified_videos": manifest.get("verified_videos") == expected_videos,
        "assigned_frames": manifest.get("assigned_frames") == expected_frames,
        "verified_frames": manifest.get("verified_frames") == expected_frames,
        "nb24": manifest.get("nb24_complete_sha256") == EXPECTED_NB24_COMPLETE_SHA256,
    }
    if pipeline == "bge_m3":
        checks.update({
            "vector_rows": manifest.get("vector_rows") == expected_frames,
            "caption_rows": manifest.get("non_empty_caption_rows") == expected_frames,
            "caption_coverage": float(manifest.get("caption_coverage", -1.0)) == 1.0,
            "parity_freeze": manifest.get("parity_freeze_sha256") == EXPECTED_PARITY_FREEZE_SHA256,
        })
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError(f"{pipeline}/{runner}: incompatible runner manifest: {failed}")


def inspect_vector_payload(
    path: Path, *, expected_ids: list[str], dim: int, norm_atol: float,
) -> np.ndarray:
    with np.load(path, allow_pickle=False) as payload:
        ids = [str(value) for value in payload["ids"].tolist()]
        vectors = np.asarray(payload["vectors"], dtype=np.float32)
    if ids != expected_ids or len(ids) != len(set(ids)):
        raise RuntimeError(f"Vector identity/order drift in {path.name}")
    if vectors.shape != (len(ids), dim) or not np.isfinite(vectors).all():
        raise RuntimeError(f"Vector shape/value drift in {path.name}: {vectors.shape}")
    norms = np.linalg.norm(vectors, axis=1)
    if len(norms) and float(np.max(np.abs(norms - 1.0))) > norm_atol:
        raise RuntimeError(f"Vector normalization drift in {path.name}")
    return vectors


def validate_text_payload(payload: Any, *, allowed_ids: set[str], artifact: str) -> dict[str, str]:
    if not isinstance(payload, dict):
        raise RuntimeError(f"{artifact}: expected JSON object")
    cleaned = {str(key): str(value).strip() for key, value in payload.items() if str(value).strip()}
    unexpected = set(cleaned) - allowed_ids
    if unexpected:
        raise RuntimeError(f"{artifact}: unexpected ids {sorted(unexpected)[:3]}")
    return cleaned


## 2 — Refresh repository, dependencies và Modal


In [ ]:
import os
import subprocess
import sys
from datetime import datetime, timezone


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1600:]}")
    return result.stdout.strip()


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
askpass = WORK / "git-askpass.sh"
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO):
            raise RuntimeError("Restart Session: Kaggle checkout has local changes")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=env)
finally:
    askpass.unlink(missing_ok=True)

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "notebooks/requirements-kaggle-keyframe-augmentation.txt")])
# FlatBundle imports the repository's BM25 adapter even though this aggregate
# only validates files.  The producer requirements intentionally omit it, so
# install the small runtime dependency explicitly for a clean Kaggle image.
run([sys.executable, "-m", "pip", "install", "-q", "rank-bm25>=0.2.2,<1"])
sys.path.insert(0, str(REPO / "src"))

import yaml
from mfe.contracts.ids import build_keyframe_id
from mfe.offline.keyframe_augmentation.common import atomic_write_json, atomic_write_jsonl, read_jsonl, sha256_file, sha256_json
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.flat_arms import FlatBundle

repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
cfg = yaml.safe_load((REPO / "config/keyframe_augmentation_kaggle.yaml").read_text(encoding="utf-8"))
research = ModalVolumeTransport(cfg["output"]["volume"], create_if_missing=False)
root = cfg["output"]["root"].rstrip("/")
nb23_root = f"{root}/gate1-feature-parity-worklist-v1"
nb24_root = f"{root}/gate1-siglip-b0-b4-screen-v1"
siglip_root = f"{root}/gate1-features-siglip-v1"
bge_root = f"{root}/gate1-b1-bge-m3-v1"
output_root = f"{root}/gate1-b1-feature-aggregate-v1"
print({"repo_commit": repo_commit, "scope": RUN_SCOPE, "device": "CPU", "output_root": output_root})


## 3 — Khóa lineage và chia đúng 5.301 frame B1


In [ ]:
freeze_path = INPUT / "parity-freeze.json"
worklist_path = INPUT / "feature-parity-worklist.jsonl"
nb24_path = INPUT / "nb24-COMPLETE.json"
research.get(f"{nb23_root}/parity-freeze.json", freeze_path)
research.get(f"{nb23_root}/feature-parity-worklist.jsonl", worklist_path)
research.get(f"{nb24_root}/COMPLETE.json", nb24_path)
if sha256_file(freeze_path) != EXPECTED_PARITY_FREEZE_SHA256:
    raise RuntimeError("Notebook 23 parity freeze SHA drift")
if sha256_file(nb24_path) != EXPECTED_NB24_COMPLETE_SHA256:
    raise RuntimeError("Notebook 24 COMPLETE SHA drift")
freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
signed_payload(freeze, "freeze_payload_hash", sha256_json)
if freeze.get("status") != "READY_FOR_FEATURE_PARITY_PRODUCERS":
    raise RuntimeError("Notebook 23 freeze is not ready")
if sha256_file(worklist_path) != freeze["files"]["feature-parity-worklist.jsonl"]:
    raise RuntimeError("Feature parity worklist SHA drift")
all_rows = list(read_jsonl(worklist_path))
b1_rows = sorted((row for row in all_rows if "B1" in row["arms"]), key=lambda row: (str(row["video_id"]), int(row["frame_idx"])))
if len(b1_rows) != EXPECTED_B1_COUNT or len({str(row["keyframe_id"]) for row in b1_rows}) != EXPECTED_B1_COUNT:
    raise RuntimeError("B1 identity/count drift")
for row in b1_rows:
    if str(row["keyframe_id"]) != build_keyframe_id(str(row["video_id"]), int(row["frame_idx"])):
        raise RuntimeError(f"B1 identity mismatch: {row['keyframe_id']}")
partitions, runner_loads = split_b1_rows(b1_rows)
if runner_loads != {"quang": 2651, "trong": 2650}:
    raise RuntimeError(f"B1 runner load drift: {runner_loads}")
rows_by_video = {video_id: rows for runner in RUNNERS for video_id, rows in partitions[runner].items()}
runner_by_video = {video_id: runner for runner in RUNNERS for video_id in partitions[runner]}
all_b1_ids = [str(row["keyframe_id"]) for row in b1_rows]
all_b1_id_set = set(all_b1_ids)
print({"lineage": "PASS", "feature_frames": len(b1_rows), "videos": len(rows_by_video), "runner_loads": runner_loads})


## 4 — Xác minh 8 runner manifest


In [ ]:
MANIFEST_LEDGER = {}
for pipeline, spec in {**TEXT_SPECS, "bge_m3": {"root": "gate1-b1-bge-m3-v1"}}.items():
    for runner in RUNNERS:
        expected = partitions[runner]
        expected_frames = sum(len(rows) for rows in expected.values())
        remote = f"{root}/{spec['root']}/runner-manifests/{runner}.json"
        local = INPUT / "runner-manifests" / pipeline / f"{runner}.json"
        research.get(remote, local)
        manifest = json.loads(local.read_text(encoding="utf-8"))
        validate_runner_manifest(
            manifest,
            pipeline=pipeline,
            runner=runner,
            expected_videos=len(expected),
            expected_frames=expected_frames,
            hash_json=sha256_json,
        )
        MANIFEST_LEDGER[f"{pipeline}/{runner}"] = {
            "remote": remote,
            "sha256": sha256_file(local),
            "manifest_payload_hash": manifest["manifest_payload_hash"],
            "videos": len(expected),
            "frames": expected_frames,
        }
print({"runner_manifests": "PASS", "count": len(MANIFEST_LEDGER), "ledger": MANIFEST_LEDGER})


## 5 — Tải, xác minh và gộp mọi per-video artifact


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed


def fetch_file(remote: str, local: Path, expected_sha: str) -> Path:
    local.parent.mkdir(parents=True, exist_ok=True)
    research.get(remote, local)
    if sha256_file(local) != expected_sha or research.sha256(remote) != expected_sha:
        raise RuntimeError(f"Artifact SHA mismatch: {remote}")
    return local


def load_signed_marker(remote: str, local: Path) -> tuple[dict[str, Any], str]:
    local.parent.mkdir(parents=True, exist_ok=True)
    research.get(remote, local)
    marker = json.loads(local.read_text(encoding="utf-8"))
    signed_payload(marker, "marker_payload_hash", sha256_json)
    return marker, str(marker["marker_payload_hash"])


def aggregate_video(video_id: str) -> dict[str, Any]:
    runner = runner_by_video[video_id]
    rows = rows_by_video[video_id]
    expected_ids = [str(row["keyframe_id"]) for row in rows]
    expected_set = set(expected_ids)
    local_root = INPUT / "videos" / video_id
    text_parts: dict[str, dict[str, str]] = {}
    stage_fingerprints: dict[str, list[dict[str, Any]]] = defaultdict(list)
    qwen_marker_hash = None

    for pipeline, spec in TEXT_SPECS.items():
        producer_root = f"{root}/{spec['root']}"
        marker_remote = f"{producer_root}/markers/{runner}/{video_id}.json"
        marker, marker_hash = load_signed_marker(marker_remote, local_root / f"marker-{pipeline}.json")
        checks = (
            marker.get("status") == "COMPLETE",
            marker.get("pipeline") == pipeline,
            marker.get("runner") == runner,
            marker.get("video_id") == video_id,
            marker.get("frame_count") == len(rows),
            marker.get("keyframe_ids_sha256") == sha256_json(expected_ids),
            marker.get("nb24_complete_sha256") == EXPECTED_NB24_COMPLETE_SHA256,
        )
        if not all(checks):
            raise RuntimeError(f"{pipeline}/{video_id}: marker contract mismatch")
        artifacts = marker.get("artifacts")
        if not isinstance(artifacts, dict) or set(artifacts) != set(spec["artifacts"]):
            raise RuntimeError(f"{pipeline}/{video_id}: unexpected artifact set")
        for artifact_name in spec["artifacts"]:
            artifact = artifacts[artifact_name]
            remote = f"{producer_root}/parts/{artifact_name}/{video_id}.json"
            if artifact.get("remote") != remote or not isinstance(artifact.get("sha256"), str):
                raise RuntimeError(f"{pipeline}/{video_id}/{artifact_name}: artifact contract mismatch")
            local = fetch_file(remote, local_root / f"{artifact_name}.json", artifact["sha256"])
            payload = validate_text_payload(
                json.loads(local.read_text(encoding="utf-8")),
                allowed_ids=expected_set,
                artifact=artifact_name,
            )
            if int(artifact.get("rows", -1)) != len(payload):
                raise RuntimeError(f"{pipeline}/{video_id}/{artifact_name}: row count mismatch")
            text_parts[artifact_name] = payload
            stage_fingerprints[artifact_name].append({"model": artifact.get("model"), "params": artifact.get("params", {})})
        if pipeline == "qwen25":
            qwen_marker_hash = marker_hash

    if set(text_parts["qwen_captions"]) != expected_set:
        raise RuntimeError(f"{video_id}: Qwen caption coverage is not complete")

    bge_marker_remote = f"{bge_root}/markers/{runner}/{video_id}.json"
    bge_marker, _ = load_signed_marker(bge_marker_remote, local_root / "marker-bge_m3.json")
    bge_checks = (
        bge_marker.get("status") == "COMPLETE",
        bge_marker.get("pipeline") == "bge_m3",
        bge_marker.get("runner") == runner,
        bge_marker.get("video_id") == video_id,
        bge_marker.get("assigned_frame_count") == len(rows),
        bge_marker.get("caption_row_count") == len(rows),
        bge_marker.get("vector_row_count") == len(rows),
        bge_marker.get("keyframe_ids_sha256") == sha256_json(expected_ids),
        bge_marker.get("qwen_marker_payload_hash") == qwen_marker_hash,
        bge_marker.get("parity_freeze_sha256") == EXPECTED_PARITY_FREEZE_SHA256,
        bge_marker.get("nb24_complete_sha256") == EXPECTED_NB24_COMPLETE_SHA256,
    )
    if not all(bge_checks):
        raise RuntimeError(f"bge_m3/{video_id}: marker contract mismatch")
    bge_artifact = bge_marker.get("artifact")
    bge_remote = f"{bge_root}/parts/dense_vectors/{video_id}.npz"
    if not isinstance(bge_artifact, dict) or bge_artifact.get("remote") != bge_remote:
        raise RuntimeError(f"bge_m3/{video_id}: artifact contract mismatch")
    bge_local = fetch_file(bge_remote, local_root / "dense_vectors.npz", str(bge_artifact.get("sha256")))
    bge_vectors = inspect_vector_payload(bge_local, expected_ids=expected_ids, dim=BGE_DIM, norm_atol=NORM_ATOL)

    source_marker_remote = str(rows[0]["source_marker"])
    source_marker, source_hash = load_signed_marker(source_marker_remote, local_root / "marker-siglip-source.json")
    if source_hash != str(rows[0]["source_marker_payload_hash"]):
        raise RuntimeError(f"{video_id}: source SigLIP marker hash mismatch")
    siglip_remote = str(rows[0]["source_siglip"])
    siglip_sha = source_marker.get("siglip_sha256")
    if not isinstance(siglip_sha, str):
        raise RuntimeError(f"{video_id}: source SigLIP SHA missing")
    siglip_local = fetch_file(siglip_remote, local_root / "source-siglip.npz", siglip_sha)
    with np.load(siglip_local, allow_pickle=False) as payload:
        source_ids = [str(value) for value in payload["ids"].tolist()]
        source_vectors = np.asarray(payload["vectors"], dtype=np.float32)
    if source_vectors.shape != (len(source_ids), SIGLIP_DIM) or len(source_ids) != len(set(source_ids)) or not np.isfinite(source_vectors).all():
        raise RuntimeError(f"{video_id}: source SigLIP shape/identity drift")
    positions = {key: index for index, key in enumerate(source_ids)}
    missing = expected_set - set(positions)
    if missing:
        raise RuntimeError(f"{video_id}: source SigLIP missing ids {sorted(missing)[:3]}")
    siglip_vectors = source_vectors[[positions[key] for key in expected_ids]]
    siglip_norms = np.linalg.norm(siglip_vectors, axis=1)
    if float(np.max(np.abs(siglip_norms - 1.0))) > NORM_ATOL:
        raise RuntimeError(f"{video_id}: source SigLIP normalization drift")

    return {
        "video_id": video_id,
        "ids": expected_ids,
        "texts": text_parts,
        "siglip": siglip_vectors,
        "bge": bge_vectors,
        "stage_fingerprints": dict(stage_fingerprints),
    }


TEXT_AGGREGATES = {name: {} for name in ("ocr_text", "paddle_ocr", "viet_ocr", "qwen_captions", "qwen_ocr")}
SIGLIP_BY_ID: dict[str, np.ndarray] = {}
BGE_BY_ID: dict[str, np.ndarray] = {}
STAGE_METADATA: dict[str, list[dict[str, Any]]] = defaultdict(list)

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    futures = {pool.submit(aggregate_video, video_id): video_id for video_id in sorted(rows_by_video)}
    for ordinal, future in enumerate(as_completed(futures), 1):
        result = future.result()
        ids = result["ids"]
        for artifact, payload in result["texts"].items():
            duplicates = set(TEXT_AGGREGATES[artifact]) & set(payload)
            if duplicates:
                raise RuntimeError(f"{artifact}: duplicate ids {sorted(duplicates)[:3]}")
            TEXT_AGGREGATES[artifact].update(payload)
        for key, vector in zip(ids, result["siglip"], strict=True):
            if key in SIGLIP_BY_ID:
                raise RuntimeError(f"Duplicate SigLIP id: {key}")
            SIGLIP_BY_ID[key] = vector
        for key, vector in zip(ids, result["bge"], strict=True):
            if key in BGE_BY_ID:
                raise RuntimeError(f"Duplicate BGE id: {key}")
            BGE_BY_ID[key] = vector
        for artifact, values in result["stage_fingerprints"].items():
            STAGE_METADATA[artifact].extend(values)
        if ordinal == 1 or ordinal % 25 == 0 or ordinal == len(futures):
            print({"aggregated_videos": ordinal, "expected": len(futures), "elapsed_minutes": round((time.perf_counter() - started) / 60, 2)}, flush=True)

if set(SIGLIP_BY_ID) != all_b1_id_set or set(BGE_BY_ID) != all_b1_id_set:
    raise RuntimeError("Dense aggregate does not exactly cover B1")
if set(TEXT_AGGREGATES["qwen_captions"]) != all_b1_id_set:
    raise RuntimeError("Qwen caption aggregate does not exactly cover B1")
if any(set(payload) - all_b1_id_set for payload in TEXT_AGGREGATES.values()):
    raise RuntimeError("Text aggregate contains non-B1 identity")
print({"artifact_download_and_validation": "PASS", "videos": len(rows_by_video), "frames": len(all_b1_ids)})


## 6 — Freeze delta flat bundle và downstream contract


In [ ]:
for artifact, payload in TEXT_AGGREGATES.items():
    atomic_write_json(FLAT / f"{artifact}.json", {key: payload[key] for key in sorted(payload)})

siglip_matrix = np.stack([SIGLIP_BY_ID[key] for key in all_b1_ids]).astype(np.float32)
bge_matrix = np.stack([BGE_BY_ID[key] for key in all_b1_ids]).astype(np.float32)
np.savez_compressed(FLAT / "siglip_vectors.npz", ids=np.asarray(all_b1_ids, dtype=np.str_), vectors=siglip_matrix)
np.savez_compressed(FLAT / "dense_vectors.npz", ids=np.asarray(all_b1_ids, dtype=np.str_), vectors=bge_matrix)
atomic_write_jsonl(OUTPUT / "b1-worklist.jsonl", b1_rows)

bundle = FlatBundle.load(OUTPUT / "delta-flat")
if bundle.vector_keys.get("siglip") != all_b1_ids or bundle.vector_keys.get("bge_m3") != all_b1_ids:
    raise RuntimeError("FlatBundle dense identity drift")
for source, artifact in {
    "ocr": "ocr_text",
    "ocr_paddle": "paddle_ocr",
    "ocr_vietocr": "viet_ocr",
    "caption_qwen": "qwen_captions",
    "ocr_qwen25": "qwen_ocr",
}.items():
    if set(bundle.texts.get(source, {})) != set(TEXT_AGGREGATES[artifact]):
        raise RuntimeError(f"FlatBundle text identity drift: {source}")

def unique_metadata(values: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_hash = {sha256_json(value): value for value in values}
    return [by_hash[key] for key in sorted(by_hash)]

artifact_paths = sorted(path for path in OUTPUT.rglob("*") if path.is_file())
artifact_ledger = {
    path.relative_to(OUTPUT).as_posix(): {"sha256": sha256_file(path), "bytes": path.stat().st_size}
    for path in artifact_paths
}
stage_metadata = {artifact: unique_metadata(values) for artifact, values in sorted(STAGE_METADATA.items())}
report = {
    "schema_version": 1,
    "status": "PASS",
    "decision_authority": "b1_delta_feature_contract_only",
    "repo_commit": repo_commit,
    "feature_frames": len(all_b1_ids),
    "source_videos": len(rows_by_video),
    "runner_loads": runner_loads,
    "text_rows": {artifact: len(payload) for artifact, payload in sorted(TEXT_AGGREGATES.items())},
    "siglip_rows": len(SIGLIP_BY_ID),
    "siglip_dim": SIGLIP_DIM,
    "bge_rows": len(BGE_BY_ID),
    "bge_dim": BGE_DIM,
    "qwen_caption_coverage": len(TEXT_AGGREGATES["qwen_captions"]) / len(all_b1_ids),
    "runner_manifests": MANIFEST_LEDGER,
    "stage_metadata": stage_metadata,
    "stage_metadata_sha256": sha256_json(stage_metadata),
    "input_lineage": {
        "parity_freeze_sha256": EXPECTED_PARITY_FREEZE_SHA256,
        "nb24_complete_sha256": EXPECTED_NB24_COMPLETE_SHA256,
        "worklist_sha256": sha256_file(worklist_path),
    },
    "delta_artifact_contract_status": "PASS",
    "baseline_numeric_parity_status": "UNVERIFIED_LEGACY_BASELINE_STAGE_METADATA",
    "snapshot_status": "DEFERRED_TO_NOTEBOOK_30_BASE_PLUS_DELTA",
    "official_ab_status": "READY_NOT_EXECUTED",
    "heldout_accessed": False,
    "btc_mutated": False,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
report["report_payload_hash"] = sha256_json(report)
atomic_write_json(OUTPUT / "aggregate-report.json", report)

# Rebuild ledger after report is written; COMPLETE signs every published file.
artifact_paths = sorted(path for path in OUTPUT.rglob("*") if path.is_file())
artifact_ledger = {
    path.relative_to(OUTPUT).as_posix(): {"sha256": sha256_file(path), "bytes": path.stat().st_size}
    for path in artifact_paths
}
complete = {
    "schema_version": 1,
    "status": "COMPLETE",
    "output_status": "READY_FOR_B0_B1_DEV_AB",
    "repo_commit": repo_commit,
    "feature_frames": len(all_b1_ids),
    "siglip_rows": len(SIGLIP_BY_ID),
    "bge_rows": len(BGE_BY_ID),
    "aggregate_report_sha256": sha256_file(OUTPUT / "aggregate-report.json"),
    "artifact_ledger": artifact_ledger,
    "artifact_ledger_sha256": sha256_json(artifact_ledger),
    "next": "Notebook 30 attaches the frozen BTC baseline, builds a disposable B0+B1 overlay/snapshot, and runs paired DEV A/B.",
    "heldout_accessed": False,
    "btc_mutated": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}
complete["manifest_payload_hash"] = sha256_json(complete)
atomic_write_json(OUTPUT / "COMPLETE.json", complete)
print({"local_bundle_validation": "PASS", "feature_frames": len(all_b1_ids), "text_rows": report["text_rows"], "siglip_rows": len(SIGLIP_BY_ID), "bge_rows": len(BGE_BY_ID)})


## 7 — Publish research-only và verify read-back


In [ ]:
complete_path = OUTPUT / "COMPLETE.json"
publish_files = sorted(
    (path for path in OUTPUT.rglob("*") if path.is_file() and path != complete_path),
    key=lambda path: path.relative_to(OUTPUT).as_posix(),
)
for ordinal, path in enumerate(publish_files, 1):
    relative = path.relative_to(OUTPUT).as_posix()
    remote = f"{output_root}/{relative}"
    research.put(path, remote)
    if research.sha256(remote) != sha256_file(path):
        raise RuntimeError(f"Remote upload verification failed: {relative}")
    if ordinal == 1 or ordinal % 5 == 0 or ordinal == len(publish_files):
        print({"published_data_files": ordinal, "expected": len(publish_files)}, flush=True)

# COMPLETE is deliberately published last so a partial upload can never look ready.
complete_remote = f"{output_root}/COMPLETE.json"
research.put(complete_path, complete_remote)
if research.sha256(complete_remote) != sha256_file(complete_path):
    raise RuntimeError("Remote COMPLETE upload verification failed")

complete_readback = INPUT / "COMPLETE-readback.json"
research.get(complete_remote, complete_readback)
if sha256_file(complete_readback) != sha256_file(complete_path):
    raise RuntimeError("Remote COMPLETE read-back SHA mismatch")
print({
    "B1_FEATURE_AGGREGATE": "COMPLETE",
    "feature_frames": len(all_b1_ids),
    "siglip_rows": len(SIGLIP_BY_ID),
    "bge_rows": len(BGE_BY_ID),
    "qwen_caption_coverage": 1.0,
    "remote_output": output_root,
    "complete_sha256": sha256_file(OUTPUT / "COMPLETE.json"),
    "next": "Notebook 30 B0-B1 paired DEV A/B",
    "official_score_claimed": False,
    "production_go_claimed": False,
})


## 8 — Cách chạy và ý nghĩa

Chạy `Save Version / Run All` với CPU và Internet bật. Notebook không cần GPU.
Không chạy nếu một trong 8 runner manifest chưa `COMPLETE`.

`COMPLETE` ở đây chỉ chứng minh delta B1 có đúng danh tính, lineage và feature
artifact cần cho mọi arm. Nó chưa chứng minh B1 tăng điểm. Notebook 30 phải attach
baseline BTC bất biến, dựng overlay B0/B1 trong runtime tạm và chạy paired DEV
A/B; chỉ kết quả đó mới có quyền kết luận uplift.
